# BERT Action Ranker Training

This notebook runs the WebQSP+CWQ action-scoring router from the reorganized router tree. It uses the offline tables under `Router_for_FeDeLiS/preprocessed/webqsp_cwq`, launches the trainer from `Router_for_FeDeLiS/scripts/webqsp_cwq`, and writes run artifacts under `Router_for_FeDeLiS/outputs/webqsp_cwq`.


In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print({'is_colab': IS_COLAB, 'python': sys.executable})

# Choose one: 'colab_git', 'colab_drive', 'local_repo'
BOOTSTRAP_MODE = 'colab_drive'

# Fill this if using BOOTSTRAP_MODE='colab_git'
GIT_REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
GIT_BRANCH = 'main'
COLAB_CLONE_DIR = Path('/content/project')

# Fill this if using BOOTSTRAP_MODE='colab_drive'
DRIVE_PROJECT_PATH = Path('/content/drive/MyDrive/MIT_nlp/project')
COLAB_DRIVE_COPY_DIR = Path('/content/project')

# Use this if BOOTSTRAP_MODE='local_repo'
LOCAL_PROJECT_ROOT = Path('/Users/tzhao/Desktop/Harvard/MIT_nlp/project')

# Single-run defaults (the sweep below overrides most of these)
RUN_NAME = 'default'
MODEL_NAME = 'distilbert-base-uncased'
FREEZE_MODE = 'last1'
MAX_LENGTH = 64
POINTWISE_BATCH_SIZE = 32
PAIRWISE_BATCH_SIZE = 32
EPOCHS = 30
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
N_SPLITS = 3
PATIENCE = 5
SEED = 42
POINTWISE_LOSS_WEIGHT = 1.0
PAIRWISE_LOSS_WEIGHT = 0.5
REWARD_COST_WEIGHT = 0.05
PAIRWISE_MARGIN_WEIGHT = 1.0
PAIRWISE_LOSS_MODE = 'weighted_logistic'
PAIRWISE_DYNAMIC_MARGIN_SCALE = 0.5
REBUILD_PREPROCESSED = False


{'is_colab': True, 'python': '/usr/bin/python3'}


In [2]:
def run(cmd, cwd=None, check=True):
    if isinstance(cmd, list):
        printable = ' '.join(str(x) for x in cmd)
    else:
        printable = cmd
    print(f'\n$ {printable}')
    return subprocess.run(cmd, cwd=cwd, check=check)


def ensure_python_packages():
    packages = [
        'torch',
        'transformers',
        'scikit-learn',
        'pandas',
        'numpy',
    ]
    run([sys.executable, '-m', 'pip', 'install', '-q', *packages])


def bootstrap_project_root() -> Path:
    if BOOTSTRAP_MODE == 'local_repo':
        project_root = LOCAL_PROJECT_ROOT
        if not project_root.exists():
            raise FileNotFoundError(f'LOCAL_PROJECT_ROOT not found: {project_root}')
        return project_root

    if not IS_COLAB:
        raise RuntimeError(
            'BOOTSTRAP_MODE is set for Colab, but this notebook is not running in Colab. ' 
            'Use local_repo or switch to a Colab runtime.'
        )

    if BOOTSTRAP_MODE == 'colab_git':
        if '<your-user>' in GIT_REPO_URL or '<your-repo>' in GIT_REPO_URL:
            raise ValueError('Set GIT_REPO_URL before running the bootstrap cell.')
        if COLAB_CLONE_DIR.exists() and any(COLAB_CLONE_DIR.iterdir()):
            print(f'Using existing clone at {COLAB_CLONE_DIR}')
        else:
            run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(COLAB_CLONE_DIR)])
        return COLAB_CLONE_DIR

    if BOOTSTRAP_MODE == 'colab_drive':
        from google.colab import drive
        drive.mount('/content/drive')
        if not DRIVE_PROJECT_PATH.exists():
            raise FileNotFoundError(f'DRIVE_PROJECT_PATH not found: {DRIVE_PROJECT_PATH}')
        if COLAB_DRIVE_COPY_DIR.exists():
            shutil.rmtree(COLAB_DRIVE_COPY_DIR)
        shutil.copytree(DRIVE_PROJECT_PATH, COLAB_DRIVE_COPY_DIR)
        return COLAB_DRIVE_COPY_DIR

    raise ValueError(f'Unknown BOOTSTRAP_MODE: {BOOTSTRAP_MODE}')


ensure_python_packages()
PROJECT_ROOT = bootstrap_project_root().resolve()
ROUTER_DIR = PROJECT_ROOT / 'Router_for_FeDeLiS'
TRAINER_PATH = ROUTER_DIR / 'scripts' / 'webqsp_cwq' / 'train_bert_action_ranker.py'
PREPROCESS_PATH = ROUTER_DIR / 'scripts' / 'webqsp_cwq' / 'preprocess_router_training_data.py'
RAW_DATA_DIR = ROUTER_DIR / 'raw_data' / 'webqsp_cwq'
PREPROCESSED_DIR = ROUTER_DIR / 'preprocessed' / 'webqsp_cwq'
OUTPUT_ROOT = ROUTER_DIR / 'outputs' / 'webqsp_cwq'
OUTPUT_DIR = OUTPUT_ROOT / f'bert_action_ranker_{RUN_NAME}'

print({
    'project_root': str(PROJECT_ROOT),
    'router_dir': str(ROUTER_DIR),
    'trainer_exists': TRAINER_PATH.exists(),
    'preprocess_exists': PREPROCESS_PATH.exists(),
    'raw_data_dir': str(RAW_DATA_DIR),
    'preprocessed_dir': str(PREPROCESSED_DIR),
})



$ /usr/bin/python3 -m pip install -q torch transformers scikit-learn pandas numpy
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'project_root': '/content/project', 'router_dir': '/content/project/Router_for_FeDeLiS', 'trainer_exists': True, 'preprocess_exists': True, 'raw_data_dir': '/content/project/Router_for_FeDeLiS/raw_data/webqsp_cwq', 'preprocessed_dir': '/content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq'}


In [3]:
required_files = [
    PREPROCESSED_DIR / 'router_query_table.jsonl',
    PREPROCESSED_DIR / 'router_action_table.jsonl',
    PREPROCESSED_DIR / 'router_pairwise_table.jsonl',
]

need_preprocess = REBUILD_PREPROCESSED or not all(path.exists() for path in required_files)
print({'need_preprocess': need_preprocess})

if need_preprocess:
    run([sys.executable, str(PREPROCESS_PATH)], cwd=PROJECT_ROOT)

for path in required_files:
    print(path, path.exists(), path.stat().st_size if path.exists() else None)


{'need_preprocess': False}
/content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl True 239121
/content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl True 4056470
/content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl True 12036863


In [4]:
import json

query_path = PREPROCESSED_DIR / 'router_query_table.jsonl'
action_path = PREPROCESSED_DIR / 'router_action_table.jsonl'
pairwise_path = PREPROCESSED_DIR / 'router_pairwise_table.jsonl'

for path in [query_path, action_path, pairwise_path]:
    with path.open('r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    print(path.name, count)

with query_path.open('r', encoding='utf-8') as f:
    sample_query = json.loads(next(f))
with action_path.open('r', encoding='utf-8') as f:
    sample_action = json.loads(next(f))
with pairwise_path.open('r', encoding='utf-8') as f:
    sample_pair = json.loads(next(f))

print('sample query keys:', sorted(sample_query.keys())[:20])
print('sample action keys:', sorted(sample_action.keys())[:20])
print('sample pair keys:', sorted(sample_pair.keys())[:20])


router_query_table.jsonl 220
router_action_table.jsonl 2345
router_pairwise_table.jsonl 16002
sample query keys: ['baseline_action_cost', 'baseline_depth', 'baseline_is_best_by_f1_cost', 'baseline_is_within_best_epsilon', 'baseline_llm_acc', 'baseline_llm_f1', 'baseline_llm_hit', 'baseline_llm_precision', 'baseline_llm_recall', 'baseline_runtime_sec', 'baseline_width', 'benchmark', 'best_action_cost', 'best_depth', 'best_llm_acc', 'best_llm_f1', 'best_llm_hit', 'best_llm_precision', 'best_llm_recall', 'best_minus_baseline_f1']
sample action keys: ['action_cost', 'action_depth', 'action_is_within_best_epsilon', 'action_width', 'baseline_depth', 'baseline_is_within_best_epsilon', 'baseline_llm_f1', 'baseline_width', 'benchmark', 'best_depth', 'best_llm_f1', 'best_width', 'direct_acc', 'direct_f1', 'direct_hit', 'direct_precision', 'direct_recall', 'f1_gain_over_baseline', 'f1_gap_to_best', 'ground_truth']
sample pair keys: ['benchmark', 'cost_margin_left_minus_right', 'f1_margin', 'groun

## Shared Sweep Runner

The next cells define the shared sweep machinery once. After that, the notebook has two separate experiment sections:
- the original pairwise objective (`weighted_logistic`)
- the new dynamic-margin objective (`dynamic_margin`)

Both sections support sweeps. The current default is still the targeted 1D sweep over `pairwise_loss_weight`.


In [6]:
import json
import pandas as pd
import time

DRIVE_SWEEP_ROOT = Path('/content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/webqsp_cwq/router_sweeps')
RESUME_IF_DRIVE_RESULT_EXISTS = True
OVERWRITE_DRIVE_RUN_DIR = True

MODEL_NAME = 'distilbert-base-uncased'
FREEZE_MODE = 'last1'
MAX_LENGTH = 64
POINTWISE_BATCH_SIZE = 32
PAIRWISE_BATCH_SIZE = 32
EPOCHS = 12
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
N_SPLITS = 3
PATIENCE = 3
SEED = 42
POINTWISE_LOSS_WEIGHT = 1.0
REWARD_COST_WEIGHT = 0.02
PAIRWISE_MARGIN_WEIGHT = 1.0

DEFAULT_PAIRWISE_LOSS_WEIGHTS = [2.0, 3.0, 4.0, 6.0]

def build_train_cmd(run_output_dir: Path, cfg: dict) -> list[str]:
    cmd = [
        sys.executable,
        str(TRAINER_PATH),
        '--query-path', str(PREPROCESSED_DIR / 'router_query_table.jsonl'),
        '--action-path', str(PREPROCESSED_DIR / 'router_action_table.jsonl'),
        '--pairwise-path', str(PREPROCESSED_DIR / 'router_pairwise_table.jsonl'),
        '--output-dir', str(run_output_dir),
        '--model-name', MODEL_NAME,
        '--freeze-mode', FREEZE_MODE,
        '--max-length', str(MAX_LENGTH),
        '--pointwise-batch-size', str(POINTWISE_BATCH_SIZE),
        '--pairwise-batch-size', str(PAIRWISE_BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--encoder-learning-rate', str(ENCODER_LR),
        '--head-learning-rate', str(HEAD_LR),
        '--weight-decay', str(WEIGHT_DECAY),
        '--dropout', str(DROPOUT),
        '--n-splits', str(N_SPLITS),
        '--early-stopping-patience', str(PATIENCE),
        '--seed', str(SEED),
        '--pointwise-loss-weight', str(POINTWISE_LOSS_WEIGHT),
        '--pairwise-loss-weight', str(cfg['pairwise_loss_weight']),
        '--reward-cost-weight', str(cfg['reward_cost_weight']),
        '--pairwise-margin-weight', str(PAIRWISE_MARGIN_WEIGHT),
        '--pairwise-loss-mode', cfg['pairwise_loss_mode'],
    ]
    if cfg['pairwise_loss_mode'] == 'dynamic_margin':
        cmd.extend([
            '--pairwise-dynamic-margin-scale', str(cfg['pairwise_dynamic_margin_scale']),
        ])
    return cmd


def load_summary_wide(summary_csv_path: Path) -> dict:
    summary_df = pd.read_csv(summary_csv_path)
    row = {rec['metric']: rec['mean'] for rec in summary_df.to_dict('records')}
    std_row = {f"{rec['metric']}__std": rec['std'] for rec in summary_df.to_dict('records')}
    row.update(std_row)
    return row


def copy_run_to_drive(local_output_dir: Path, drive_run_dir: Path) -> None:
    if drive_run_dir.exists() and OVERWRITE_DRIVE_RUN_DIR:
        shutil.rmtree(drive_run_dir)
    drive_run_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_run_dir)


def write_drive_aggregate(rows: list[dict], drive_group_dir: Path) -> None:
    aggregate_df = pd.DataFrame(rows)
    aggregate_df = aggregate_df.sort_values(['exact_accuracy', 'strategy_macro_f1'], ascending=[False, False])
    aggregate_csv = drive_group_dir / 'sweep_summary.csv'
    aggregate_jsonl = drive_group_dir / 'sweep_summary.jsonl'
    aggregate_df.to_csv(aggregate_csv, index=False)
    with aggregate_jsonl.open('w', encoding='utf-8') as handle:
        for rec in aggregate_df.to_dict('records'):
            handle.write(json.dumps(rec, ensure_ascii=False) + '\n')


def build_sweep_configs(
    *,
    pairwise_loss_mode: str,
    sweep_group_name: str,
    pairwise_loss_weights: list[float],
    reward_cost_weight: float,
    dynamic_margin_scales: list[float] | None = None,
) -> list[dict]:
    configs: list[dict] = []
    if pairwise_loss_mode == 'dynamic_margin':
        dynamic_margin_scales = dynamic_margin_scales or [0.5]
        for margin_scale in dynamic_margin_scales:
            for pairwise_loss_weight in pairwise_loss_weights:
                run_name = (
                    f'{pairwise_loss_mode}_'
                    f'pair{str(pairwise_loss_weight).replace(".", "p")}_'
                    f'cost{str(reward_cost_weight).replace(".", "p")}_'
                    f'mscale{str(margin_scale).replace(".", "p")}_'
                    f'{FREEZE_MODE}'
                )
                configs.append({
                    'sweep_group': sweep_group_name,
                    'run_name': run_name,
                    'pairwise_loss_mode': pairwise_loss_mode,
                    'pairwise_loss_weight': pairwise_loss_weight,
                    'reward_cost_weight': reward_cost_weight,
                    'pairwise_dynamic_margin_scale': margin_scale,
                })
    else:
        for pairwise_loss_weight in pairwise_loss_weights:
            run_name = (
                f'{pairwise_loss_mode}_'
                f'pair{str(pairwise_loss_weight).replace(".", "p")}_'
                f'cost{str(reward_cost_weight).replace(".", "p")}_'
                f'{FREEZE_MODE}'
            )
            configs.append({
                'sweep_group': sweep_group_name,
                'run_name': run_name,
                'pairwise_loss_mode': pairwise_loss_mode,
                'pairwise_loss_weight': pairwise_loss_weight,
                'reward_cost_weight': reward_cost_weight,
                'pairwise_dynamic_margin_scale': None,
            })
    return configs


def run_one_config(cfg: dict, drive_group_dir: Path) -> dict:
    run_name = cfg['run_name']
    local_output_dir = OUTPUT_ROOT / f"bert_action_ranker_{cfg['sweep_group']}_{run_name}"
    drive_run_dir = drive_group_dir / local_output_dir.name
    drive_summary_path = drive_run_dir / 'bert_action_ranker_cv_summary.csv'

    if RESUME_IF_DRIVE_RESULT_EXISTS and drive_summary_path.exists():
        print(f'Skipping existing Drive result: {run_name}')
        wide = load_summary_wide(drive_summary_path)
        result = {
            'sweep_group': cfg['sweep_group'],
            'run_name': run_name,
            'status': 'skipped_existing',
            'freeze_mode': FREEZE_MODE,
            'model_name': MODEL_NAME,
            'epochs': EPOCHS,
            'n_splits': N_SPLITS,
            'pairwise_loss_mode': cfg['pairwise_loss_mode'],
            'pairwise_dynamic_margin_scale': cfg['pairwise_dynamic_margin_scale'],
            'pairwise_loss_weight': cfg['pairwise_loss_weight'],
            'reward_cost_weight': cfg['reward_cost_weight'],
        }
        result.update(wide)
        return result

    if local_output_dir.exists():
        shutil.rmtree(local_output_dir)
    local_output_dir.mkdir(parents=True, exist_ok=True)

    cmd = build_train_cmd(local_output_dir, cfg)
    start_time = time.time()
    run(cmd, cwd=PROJECT_ROOT)
    elapsed_sec = time.time() - start_time

    summary_csv = local_output_dir / 'bert_action_ranker_cv_summary.csv'
    if not summary_csv.exists():
        raise FileNotFoundError(f'Missing summary CSV for {run_name}: {summary_csv}')

    copy_run_to_drive(local_output_dir, drive_run_dir)
    wide = load_summary_wide(summary_csv)
    result = {
        'sweep_group': cfg['sweep_group'],
        'run_name': run_name,
        'status': 'completed',
        'freeze_mode': FREEZE_MODE,
        'model_name': MODEL_NAME,
        'epochs': EPOCHS,
        'n_splits': N_SPLITS,
        'pairwise_loss_mode': cfg['pairwise_loss_mode'],
        'pairwise_dynamic_margin_scale': cfg['pairwise_dynamic_margin_scale'],
        'pairwise_loss_weight': cfg['pairwise_loss_weight'],
        'reward_cost_weight': cfg['reward_cost_weight'],
        'elapsed_sec': elapsed_sec,
    }
    result.update(wide)

    metadata = {
        'project_root': str(PROJECT_ROOT),
        'trainer_path': str(TRAINER_PATH),
        'preprocessed_dir': str(PREPROCESSED_DIR),
        'output_dir': str(local_output_dir),
        'drive_run_dir': str(drive_run_dir),
        'command': cmd,
        'config': result,
    }
    with (drive_run_dir / 'run_metadata.json').open('w', encoding='utf-8') as handle:
        json.dump(metadata, handle, indent=2)

    print(f'Finished {run_name} in {elapsed_sec / 60.0:.2f} minutes')
    return result


def execute_sweep(sweep_configs: list[dict]) -> tuple[Path, list[dict]]:
    if not sweep_configs:
        raise ValueError('sweep_configs is empty')
    sweep_group_name = sweep_configs[0]['sweep_group']
    drive_group_dir = DRIVE_SWEEP_ROOT / sweep_group_name
    drive_group_dir.mkdir(parents=True, exist_ok=True)

    sweep_results: list[dict] = []
    for cfg in sweep_configs:
        result = run_one_config(cfg, drive_group_dir)
        sweep_results.append(result)
        write_drive_aggregate(sweep_results, drive_group_dir)
        display(
            pd.DataFrame(sweep_results)
            .sort_values(['exact_accuracy', 'strategy_macro_f1'], ascending=[False, False])
            .reset_index(drop=True)
        )

    print(f'Sweep outputs saved under: {drive_group_dir}')
    return drive_group_dir, sweep_results


def show_leaderboard(drive_group_dir: Path):
    aggregate_csv = drive_group_dir / 'sweep_summary.csv'
    aggregate_df = pd.read_csv(aggregate_csv)
    ranking_columns = [
        'run_name',
        'pairwise_loss_mode',
        'pairwise_dynamic_margin_scale',
        'pairwise_loss_weight',
        'reward_cost_weight',
        'exact_accuracy',
        'strategy_macro_f1',
        'depth_accuracy',
        'width_accuracy',
        'mean_pred_minus_baseline_f1',
        'mean_best_minus_pred_f1',
        'cost_sensitive_error',
        'elapsed_sec',
        'status',
    ]
    available_columns = [col for col in ranking_columns if col in aggregate_df.columns]
    aggregate_df = aggregate_df[available_columns].sort_values(
        ['exact_accuracy', 'strategy_macro_f1'],
        ascending=[False, False],
    ).reset_index(drop=True)
    print(f'Leaderboard for {drive_group_dir.name}')
    display(aggregate_df)


## Section 1: Original Pairwise Objective

This section uses the original `weighted_logistic` pairwise objective. It still supports sweeps; by default it runs the same targeted 1D sweep over `pairwise_loss_weight`.


In [7]:
ORIGINAL_SWEEP_GROUP_NAME = 'pair_weight_weighted_logistic_v1'
ORIGINAL_PAIRWISE_LOSS_WEIGHTS = DEFAULT_PAIRWISE_LOSS_WEIGHTS
ORIGINAL_REWARD_COST_WEIGHT = REWARD_COST_WEIGHT

original_sweep_configs = build_sweep_configs(
    pairwise_loss_mode='weighted_logistic',
    sweep_group_name=ORIGINAL_SWEEP_GROUP_NAME,
    pairwise_loss_weights=ORIGINAL_PAIRWISE_LOSS_WEIGHTS,
    reward_cost_weight=ORIGINAL_REWARD_COST_WEIGHT,
)

display(pd.DataFrame(original_sweep_configs))


,sweep_group,run_name,pairwise_loss_mode,pairwise_loss_weight,reward_cost_weight,pairwise_dynamic_margin_scale
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_cost0p02_last1,weighted_logistic,2.0,0.02,None
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair3p0_cost0p02_last1,weighted_logistic,3.0,0.02,None
2,pair_weight_weighted_logistic_v1,weighted_logistic_pair4p0_cost0p02_last1,weighted_logistic,4.0,0.02,None
3,pair_weight_weighted_logistic_v1,weighted_logistic_pair6p0_cost0p02_last1,weighted_logistic,6.0,0.02,None


In [8]:
original_drive_group_dir, original_sweep_results = execute_sweep(original_sweep_configs)



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair2p0_cost0p02_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finishe

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,2.0,...,0.02935,0.098587,0.054632,0.104951,0.071633,0.055669,0.011442,0.006695,0.046463,0.049742



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair3p0_cost0p02_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 3.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finishe

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,2.0,...,0.029350,0.098587,0.054632,0.104951,0.071633,0.055669,0.011442,0.006695,0.046463,0.049742
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair3p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,3.0,...,0.023035,0.118170,0.078652,0.156166,0.142080,0.061948,0.011442,0.006695,0.055434,0.057332



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair4p0_cost0p02_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 4.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finishe

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,2.0,...,0.029350,0.098587,0.054632,0.104951,0.071633,0.055669,0.011442,0.006695,0.046463,0.049742
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair4p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,4.0,...,0.044215,0.047085,0.069637,0.066880,0.325536,0.051123,0.011442,0.006695,0.043683,0.046006
2,pair_weight_weighted_logistic_v1,weighted_logistic_pair3p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,3.0,...,0.023035,0.118170,0.078652,0.156166,0.142080,0.061948,0.011442,0.006695,0.055434,0.057332



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair6p0_cost0p02_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 6.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finishe

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,2.0,...,0.029350,0.098587,0.054632,0.104951,0.071633,0.055669,0.011442,0.006695,0.046463,0.049742
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair4p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,4.0,...,0.044215,0.047085,0.069637,0.066880,0.325536,0.051123,0.011442,0.006695,0.043683,0.046006
2,pair_weight_weighted_logistic_v1,weighted_logistic_pair6p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,6.0,...,0.017570,0.140254,0.089969,0.184906,0.115185,0.032059,0.011442,0.006695,0.029090,0.029081
3,pair_weight_weighted_logistic_v1,weighted_logistic_pair3p0_cost0p02_last1,completed,last1,distilbert-base-uncased,12,3,weighted_logistic,None,3.0,...,0.023035,0.118170,0.078652,0.156166,0.142080,0.061948,0.011442,0.006695,0.055434,0.057332


Sweep outputs saved under: /content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/webqsp_cwq/router_sweeps/pair_weight_weighted_logistic_v1


In [9]:
show_leaderboard(original_drive_group_dir)


Leaderboard for pair_weight_weighted_logistic_v1


,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,reward_cost_weight,exact_accuracy,strategy_macro_f1,depth_accuracy,width_accuracy,mean_pred_minus_baseline_f1,mean_best_minus_pred_f1,cost_sensitive_error,elapsed_sec,status
0,weighted_logistic_pair2p0_cost0p02_last1,weighted_logistic,NaN,2.0,0.02,0.538584,0.165691,0.661734,0.684461,0.126973,0.233181,0.899930,833.927102,completed
1,weighted_logistic_pair4p0_cost0p02_last1,weighted_logistic,NaN,4.0,0.02,0.538407,0.176379,0.638654,0.699789,0.127915,0.232239,0.900370,771.250790,completed
2,weighted_logistic_pair6p0_cost0p02_last1,weighted_logistic,NaN,6.0,0.02,0.538231,0.218503,0.630374,0.707364,0.116581,0.243573,0.908827,679.704090,completed
3,weighted_logistic_pair3p0_cost0p02_last1,weighted_logistic,NaN,3.0,0.02,0.530832,0.188364,0.622974,0.676885,0.128910,0.231244,0.965733,588.846601,completed


## Section 2: Dynamic-Margin Pairwise Objective

This section uses the new `dynamic_margin` objective. It also supports sweeps. By default it sweeps the same `pairwise_loss_weight` values at a fixed `pairwise_dynamic_margin_scale`; you can turn that into a 2D sweep by passing multiple margin scales.


In [14]:
DYNAMIC_SWEEP_GROUP_NAME = 'pair_weight_dynamic_margin_v1'
DEFAULT_PAIRWISE_LOSS_WEIGHTS = [0.5, 1.0, 2.0]
DYNAMIC_MARGIN_SCALES = [0.5]
REWARD_COST_WEIGHT = 0.02
DYNAMIC_PAIRWISE_LOSS_WEIGHTS = DEFAULT_PAIRWISE_LOSS_WEIGHTS
DYNAMIC_REWARD_COST_WEIGHT = REWARD_COST_WEIGHT


dynamic_sweep_configs = build_sweep_configs(
    pairwise_loss_mode='dynamic_margin',
    sweep_group_name=DYNAMIC_SWEEP_GROUP_NAME,
    pairwise_loss_weights=DYNAMIC_PAIRWISE_LOSS_WEIGHTS,
    reward_cost_weight=DYNAMIC_REWARD_COST_WEIGHT,
    dynamic_margin_scales=DYNAMIC_MARGIN_SCALES,
)

display(pd.DataFrame(dynamic_sweep_configs))


,sweep_group,run_name,pairwise_loss_mode,pairwise_loss_weight,reward_cost_weight,pairwise_dynamic_margin_scale
0,pair_weight_dynamic_margin_v1,dynamic_margin_pair0p5_cost0p02_mscale0p5_last1,dynamic_margin,0.5,0.02,0.5
1,pair_weight_dynamic_margin_v1,dynamic_margin_pair1p0_cost0p02_mscale0p5_last1,dynamic_margin,1.0,0.02,0.5
2,pair_weight_dynamic_margin_v1,dynamic_margin_pair2p0_cost0p02_mscale0p5_last1,dynamic_margin,2.0,0.02,0.5


In [15]:
dynamic_drive_group_dir, dynamic_sweep_results = execute_sweep(dynamic_sweep_configs)



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_dynamic_margin_v1_dynamic_margin_pair0p5_cost0p02_mscale0p5_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 0.5 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pair

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_dynamic_margin_v1,dynamic_margin_pair0p5_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,0.5,...,0.113845,0.072287,0.034949,0.08947,0.025049,0.05418,0.011442,0.006695,0.055914,0.054135



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_dynamic_margin_v1_dynamic_margin_pair1p0_cost0p02_mscale0p5_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 1.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pair

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_dynamic_margin_v1,dynamic_margin_pair0p5_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,0.5,...,0.113845,0.072287,0.034949,0.089470,0.025049,0.054180,0.011442,0.006695,0.055914,0.054135
1,pair_weight_dynamic_margin_v1,dynamic_margin_pair1p0_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,1.0,...,0.048543,0.171700,0.065069,0.203801,0.050293,0.033115,0.011442,0.006695,0.024774,0.027467



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/webqsp_cwq/train_bert_action_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/webqsp_cwq/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/webqsp_cwq/bert_action_ranker_pair_weight_dynamic_margin_v1_dynamic_margin_pair2p0_cost0p02_mscale0p5_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 64 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.02 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pair

,sweep_group,run_name,status,freeze_mode,model_name,epochs,n_splits,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,...,strategy_macro_f1__std,depth_abs_error__std,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_llm_f1__std,mean_best_llm_f1__std,mean_baseline_llm_f1__std,mean_best_minus_pred_f1__std,mean_pred_minus_baseline_f1__std
0,pair_weight_dynamic_margin_v1,dynamic_margin_pair0p5_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,0.5,...,0.113845,0.072287,0.034949,0.089470,0.025049,0.054180,0.011442,0.006695,0.055914,0.054135
1,pair_weight_dynamic_margin_v1,dynamic_margin_pair2p0_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,2.0,...,0.096697,0.166873,0.145170,0.233507,0.031940,0.075386,0.011442,0.006695,0.081208,0.077952
2,pair_weight_dynamic_margin_v1,dynamic_margin_pair1p0_cost0p02_mscale0p5_last1,completed,last1,distilbert-base-uncased,12,3,dynamic_margin,0.5,1.0,...,0.048543,0.171700,0.065069,0.203801,0.050293,0.033115,0.011442,0.006695,0.024774,0.027467


Sweep outputs saved under: /content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/webqsp_cwq/router_sweeps/pair_weight_dynamic_margin_v1


In [16]:
show_leaderboard(dynamic_drive_group_dir)


Leaderboard for pair_weight_dynamic_margin_v1


,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pairwise_loss_weight,reward_cost_weight,exact_accuracy,strategy_macro_f1,depth_accuracy,width_accuracy,mean_pred_minus_baseline_f1,mean_best_minus_pred_f1,cost_sensitive_error,elapsed_sec,status
0,dynamic_margin_pair0p5_cost0p02_mscale0p5_last1,dynamic_margin,0.5,0.5,0.02,0.591966,0.301628,0.707717,0.684461,0.174082,0.186072,0.761716,1038.518108,completed
1,dynamic_margin_pair2p0_cost0p02_mscale0p5_last1,dynamic_margin,0.5,2.0,0.02,0.584214,0.257750,0.660853,0.715116,0.157913,0.202241,0.824612,933.901697,completed
2,dynamic_margin_pair1p0_cost0p02_mscale0p5_last1,dynamic_margin,0.5,1.0,0.02,0.576638,0.253564,0.637949,0.699965,0.159700,0.200454,0.882135,837.094223,completed


## Notes

Use the first section for the original objective and the second section for the dynamic-margin objective. Both sections are sweep-ready. The current default remains the efficient 1D pair-weight sweep.

Practical rule:
- keep `N_SPLITS = 3` for sweep stages;
- use `N_SPLITS = 5`, `EPOCHS = 30`, and `PATIENCE = 5` only for confirmation of the best one or two settings;
- to turn the dynamic-margin section into a 2D sweep, set `DYNAMIC_MARGIN_SCALES` to multiple values.
